# 02 — Controlled raster and FFT feature exploration

This is descriptive analysis, not provenance proof. It visualises the exact H1-N pixel contract: decode to RGB, crop a source-coordinate square without padding, resize once to 128 × 128, then pass the same raster either to RGB or to the FFT-magnitude transform. It does **not** reproduce the legacy D0 direct-resize radial result.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from ai_image_detector.features import (
    CONTROLLED_IMAGE_SIZE,
    fft_magnitude,
    radial_power_spectrum,
    source_normalized_rasterize,
)
from ai_image_detector.manifest import load_manifest

frame = load_manifest(Path('../data/processed/defactify_grouped/manifest.csv'), check_paths=True)
train = frame.loc[frame.split == 'train'].copy()

for group, candidate in train.groupby('leakage_group', sort=True):
    if {0, 1}.issubset(set(candidate.label)):
        chosen_group = group
        break
else:
    raise RuntimeError('No real/fake paired group was found in the grouped train split.')

examples = (
    train.loc[train.leakage_group == chosen_group]
    .sort_values(['label', 'generator'])
    .groupby('generator', as_index=False, group_keys=False)
    .head(1)
    .reset_index(drop=True)
)
examples[['leakage_group', 'label', 'generator', 'width', 'height', 'path']]

In [ ]:
fig, axes = plt.subplots(len(examples), 4, figsize=(16, 4 * len(examples)))
axes = np.atleast_2d(axes)

for row_axes, (_, row) in zip(axes, examples.iterrows(), strict=True):
    original = Image.open(row.path).convert('RGB')
    raster = source_normalized_rasterize(original, size=CONTROLLED_IMAGE_SIZE, train=False)
    magnitude = fft_magnitude(raster, size=CONTROLLED_IMAGE_SIZE)
    radial = radial_power_spectrum(raster, size=CONTROLLED_IMAGE_SIZE)

    assert raster.size == (CONTROLLED_IMAGE_SIZE, CONTROLLED_IMAGE_SIZE)
    row_axes[0].imshow(original)
    row_axes[0].set_title(f'original: {row.width}×{row.height}')
    row_axes[1].imshow(raster)
    row_axes[1].set_title('H1-N centre-crop then 128×128')
    row_axes[2].imshow(magnitude, cmap='magma')
    row_axes[2].set_title('FFT magnitude of common raster')
    row_axes[3].plot(radial)
    row_axes[3].set_title('radial spectrum of common raster')
    for axis in row_axes[:3]:
        axis.set_axis_off()

plt.tight_layout()

## What is controlled, and what is not

Training uses a seeded random square crop; validation, exploratory internal test, robustness, and external evaluation use the deterministic centre crop shown above. Letterboxing and direct rectangular-to-square resizing are prohibited because they expose geometry or interpolation as a possible label cue.

Visible spectral patterns remain hypothesis-generating. H1-N asks whether an FFT-magnitude ResNet-50 outperforms an equal-capacity RGB ResNet-50 under this shared rasterisation, across three predeclared seeds. A plot, a legacy radial score, or an individual softmax output does not establish an image's origin.